In [17]:
import os

import pandas as pd

In [18]:
topk = 0.2

In [19]:
querynamelist = [
    "Arabidopsis_thaliana",
    # "Erigeron_breviscapus",
    # "Glycine_max",
    # "Zea_mays",
    # "sheetall",
]

In [20]:
def count_lines(file_path):
    with open(file_path, "r") as file:
        lines = file.readlines()
        return len(lines)

In [ ]:
def enzypick_processing(indataframe, indexs):
    zeros_df = pd.DataFrame([{"DL Model Score": 0}] * (maxlen - 1 - len(indataframe)))
    cachedataframe = pd.concat([indataframe, zeros_df], ignore_index=True)
    cachedataframe = cachedataframe.sort_values("DL Model Score", ascending=False)
    thresholds = cachedataframe["DL Model Score"].quantile(1 - topk)
    righnum = cachedataframe[
        (cachedataframe["Upload Index"].isin(indexs))
        & (cachedataframe["DL Model Score"] > thresholds)
    ]
    allnum = indexs
    return len(righnum), len(allnum), cachedataframe

In [ ]:
with open(f"../output/enzypick_{topk}.csv", "w") as f:
    for queryname in querynamelist:
        maxlen = count_lines(f"../others/enzypick/{queryname}_enzyme.csv")
        directory_path = f"../others/enzypick/{queryname}_out/"
        dfrefer = pd.read_csv(
            f"../others/enzypick/{queryname}_sp.csv",
            header=None,
            names=["s", "p", "subname", "index"],
        )
        csv_files = sorted(
            [f for f in os.listdir(directory_path) if f.endswith(".csv")],
            key=lambda x: int(os.path.splitext(x)[0]),
        )
        allrighnum, allallnum, sumsumnum = 0, 0, 0
        final_df = pd.DataFrame()
        for num, file in enumerate(csv_files):
            file_path = os.path.join(directory_path, file)
            df = pd.read_csv(file_path)
            dfpos = dfrefer.loc[num, "index"].replace("[", "").replace("]", "")
            # print(dfpos)
            dfpos = [int(x) for x in dfpos.split("_")]
            righnum, allnum, nowdf = enzypick_processing(df, dfpos)
            # print(f"{file} {righnum} {allnum}\n")
            allrighnum += righnum
            allallnum += allnum
            sumsumnum += len(nowdf)
            testdf = df.copy()
            testdf["substrate"] = dfrefer.loc[num, "subname"]
            final_df = pd.concat([final_df, testdf], ignore_index=True)
        final_df.to_csv(f"../out4cj/enzypick_{queryname}_{topk}.csv", index=False)
        print(f"{queryname},{allrighnum},{allallnum},{sumsumnum}\n")
        f.write(f"{queryname},{allrighnum},{allallnum}\n")

Arabidopsis_thaliana,7,30,5405



In [23]:
final_df

,Upload Index,DL Model Score,Sequence,Attention,substrate
0,135,0.9349,MFSLNMRTEIESLWVFALASKFNIYMQQHFASLLVAIAITWFTITI...,"[[50, 0.6], [51, 0.8], [52, 0.7], [64, 0.6], [...",KEO
1,217,0.9344,MSLFSFPISTELLPWLLLLLIPPLLIFFLLRSPKNLPPGPPRLPIL...,"[[37, 1.0], [118, 0.7], [119, 0.6], [124, 0.6]...",KEO
2,36,0.9337,MMMKISFNTCFQILLGFIVFIASITLLGRIFSRPSKTKDRCRQLPP...,"[[14, 0.6], [15, 0.8], [445, 0.7], [446, 0.7],...",KEO
3,136,0.9321,MDLTDVIIFLFALYFINLWWRRYFSAGSSQCSLNIPPGPKGWPLVG...,"[[36, 0.8], [173, 1.0], [448, 0.6]]",KEO
4,87,0.9320,MDTLASNSSDLTTKSSLGMSSFTNMYLLTTLQALAALCFLMILNKI...,"[[59, 0.9], [61, 1.0]]",KEO
...,...,...,...,...,...
4106,111,0.7911,MNLFWNTGFCVIVLVVARVGHWWYQWSNPKSNGKLPPGSMGFPIIG...,"[[6, 0.9], [35, 0.6], [36, 0.6], [70, 0.7], [7...",BSI
4107,75,0.7728,MDYYLNNVIFSVVVTFSITLNIVFLIKSVVARFLGRRKKLPPCPRG...,"[[183, 0.9], [452, 0.7], [496, 0.8], [542, 1.0]]",BSI
4108,181,0.7117,MEETNIRVVLYSIFSLIFLIISFKFLKPKKQNLPPSPPGWLPIIGH...,"[[196, 0.7], [221, 0.6], [222, 0.7], [270, 1.0]]",BSI
4109,206,0.6750,MAESAGESYRLLSVSSSTTFLAFIIIFLLAGIARRKRRAPHRLPPG...,"[[3, 0.9], [29, 0.6], [60, 0.6], [61, 0.9], [1...",BSI
